#**KNOWLEDGE GRAPH EXPLOITATION**

This is the notebook associated with the assignment **Knowledge Graph Exploitation**. In LearnSQL you can find the assignment statament, so please carefully read it while following the scripts, answering all the questions and compleating the tasks. Remember to change the **Runtime** settings to include GPU access.

Before starting, we will install and import the required libraries:

In [ ]:
import os
from pathlib import Path

# Keep PyKEEN/PyStow cache files inside this project folder.
# This avoids permission issues on machines where the home directory is locked down.
os.environ.setdefault("PYSTOW_HOME", str(Path.cwd() / ".pystow"))
os.environ.setdefault("PYKEEN_HOME", str(Path.cwd() / ".pykeen"))

In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install pykeen

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## **Creating the Knowledge Graph**

In [ ]:
import numpy as np
from pykeen.triples import TriplesFactory
from torch_geometric.datasets import Planetoid


dataset = Planetoid(root='data/PubMed', name='PubMed')
data = dataset[0]

edge_index = data.edge_index

# Convert to numpy
heads = edge_index[0].cpu().numpy()
tails = edge_index[1].cpu().numpy()

# Create string triples
triples = np.array([
    [f"paper_{h}", "cites", f"paper_{t}"]
    for h, t in zip(heads, tails)
])

tf = TriplesFactory.from_labeled_triples(triples)

training, testing, validation = tf.split([0.8, 0.1, 0.1],random_state=2026)

## **A.1 The Most Basic Model**




In [ ]:
from pykeen.pipeline import pipeline

seed = 2026
torch.manual_seed(seed)
np.random.seed(seed)

transe_result = pipeline(
    training=training,
    testing=testing,
    validation=validation,
    model="TransE",
    model_kwargs=dict(
        embedding_dim=64,
        scoring_fct_norm=2,
    ),
    training_loop="sLCWA",
    negative_sampler="basic",
    negative_sampler_kwargs=dict(num_negs_per_pos=5),
    optimizer="Adam",
    optimizer_kwargs=dict(lr=1e-2),
    training_kwargs=dict(
        num_epochs=50,
        batch_size=1024,
    ),
    evaluator="RankBasedEvaluator",
    random_seed=seed,
    device=device,
)

transe_model = transe_result.model
print("Trained TransE with", transe_model.num_entities, "entities and", transe_model.num_relations, "relation(s).")

TransE represents each fact as a translation. For a citation triple `(paper_h, cites, paper_t)`, the model tries to make

`embedding(paper_h) + embedding(cites) ~= embedding(paper_t)`.

If we choose a paper `p` and look at the works it cites, then a plausible embedding for another paper that cites similar works is the vector that, after adding the `cites` relation vector, lands near those cited papers. With the L2 scoring norm used above, a simple estimate is:

`synthetic_paper = mean(embedding(cited_work) - embedding(cites))`.

In [ ]:
# Extract entity/relation embeddings from the trained PyKEEN model.
# PyKEEN stores embeddings by internal integer IDs, so always use the TriplesFactory mappings.
entity_embeddings = transe_model.entity_representations[0](indices=None).detach().cpu()
relation_embeddings = transe_model.relation_representations[0](indices=None).detach().cpu()

entity_to_id = training.entity_to_id
id_to_entity = {entity_id: entity for entity, entity_id in entity_to_id.items()}
cites_relation_id = training.relation_to_id["cites"]
cites_embedding = relation_embeddings[cites_relation_id]

# Store the embeddings in paper index order so they can be reused later as node features.
paper_labels = [f"paper_{i}" for i in range(data.num_nodes)]
paper_ids = torch.as_tensor([entity_to_id[label] for label in paper_labels], dtype=torch.long)
transe_embeddings_by_paper = entity_embeddings[paper_ids]
print("Embeddings in paper order:", transe_embeddings_by_paper.shape)

In [ ]:
# Choose a concrete paper from the KG. We pick the paper with the largest number of outgoing
# citation edges, because it gives a richer set of cited works for this small reasoning exercise.
edge_heads = edge_index[0].cpu()
edge_tails = edge_index[1].cpu()
out_degree = torch.bincount(edge_heads, minlength=data.num_nodes)
chosen_paper_idx = int(torch.argmax(out_degree).item())
chosen_paper_label = f"paper_{chosen_paper_idx}"

cited_paper_indices = torch.unique(edge_tails[edge_heads == chosen_paper_idx]).tolist()
cited_paper_labels = [f"paper_{idx}" for idx in cited_paper_indices]
cited_entity_ids = torch.as_tensor([entity_to_id[label] for label in cited_paper_labels], dtype=torch.long)
cited_embeddings = entity_embeddings[cited_entity_ids]

# Most probable synthetic head embedding for a paper citing similar works.
synthetic_paper_embedding = (cited_embeddings - cites_embedding).mean(dim=0)

# Retrieve the closest actual paper. The original paper will often be the closest point because
# its citation pattern is the one used to define the synthetic vector, so we also report the
# nearest different paper as the more interesting recommendation.
distances = torch.linalg.vector_norm(transe_embeddings_by_paper - synthetic_paper_embedding, ord=2, dim=1)
closest_including_chosen_idx = int(torch.argmin(distances).item())
closest_including_chosen_label = f"paper_{closest_including_chosen_idx}"

distances_excluding_chosen = distances.clone()
distances_excluding_chosen[chosen_paper_idx] = float("inf")
closest_paper_idx = int(torch.argmin(distances_excluding_chosen).item())
closest_paper_label = f"paper_{closest_paper_idx}"

print("Chosen paper:", chosen_paper_label)
print("Number of cited works used:", len(cited_paper_labels))
print("Closest actual paper to the synthetic vector:", closest_including_chosen_label)
print("Distance:", float(distances[closest_including_chosen_idx]))
print("Closest different paper:", closest_paper_label)
print("Distance:", float(distances_excluding_chosen[closest_paper_idx]))
print("First cited papers:", cited_paper_labels[:10])

In [ ]:
import matplotlib.pyplot as plt

def pca_2d(matrix):
    """Small PCA helper to avoid depending on sklearn for this plot."""
    matrix = np.asarray(matrix, dtype=np.float64)
    centered = matrix - matrix.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ vt[:2].T, vt[:2]

chosen_embedding = transe_embeddings_by_paper[chosen_paper_idx]
closest_embedding = transe_embeddings_by_paper[closest_paper_idx]
expected_cited_centroid = synthetic_paper_embedding + cites_embedding

plot_matrix = torch.vstack([
    chosen_embedding,
    synthetic_paper_embedding,
    expected_cited_centroid,
    closest_embedding,
    cited_embeddings,
]).numpy()
coords, components = pca_2d(plot_matrix)
relation_2d = cites_embedding.numpy() @ components.T

chosen_xy = coords[0]
synthetic_xy = coords[1]
expected_tail_xy = coords[2]
closest_xy = coords[3]
cited_xy = coords[4:]

plt.figure(figsize=(8, 6))
plt.scatter(cited_xy[:, 0], cited_xy[:, 1], s=35, alpha=0.55, label="works cited by chosen paper")
plt.scatter(*chosen_xy, s=130, marker="o", label=f"chosen: {chosen_paper_label}")
plt.scatter(*synthetic_xy, s=170, marker="X", label="synthetic paper vector")
plt.scatter(*expected_tail_xy, s=130, marker="P", label="synthetic + cites")
plt.scatter(*closest_xy, s=150, marker="*", label=f"nearest actual: {closest_paper_label}")

plt.arrow(
    synthetic_xy[0], synthetic_xy[1],
    relation_2d[0], relation_2d[1],
    width=0.002,
    length_includes_head=True,
    alpha=0.8,
)
plt.annotate("+ cites", xy=(expected_tail_xy[0], expected_tail_xy[1]), xytext=(5, 5), textcoords="offset points")

for xy in cited_xy[:20]:
    plt.plot([expected_tail_xy[0], xy[0]], [expected_tail_xy[1], xy[1]], linestyle=":", linewidth=0.7, alpha=0.35)

plt.title("2D sketch of TransE citation reasoning")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.legend(loc="best")
plt.grid(alpha=0.2)
plt.show()

## **A.2 Improving TransE**

In [ ]:
# Find the requested paper here

## **A.3 Training KGEs**

In [ ]:
# Train the different KGE models

## **A.4 Negative Sampling**

In [ ]:
# Obtain the corruption probabilities here

## **Loading the full PubMed Dataset**

In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='data/PubMed', name='PubMed')
data = dataset[0]

print(data)
print("Number of node features:", dataset.num_node_features)
print("Number of classes:", dataset.num_classes)

data = data.to('cuda' if torch.cuda.is_available() else 'cpu')

## **B.1: Let's Forget About the Graph**

In [ ]:
import numpy as np

X = data.x.cpu().numpy()
y = data.y.cpu().numpy()

train_mask = data.train_mask.cpu().numpy()
test_mask = data.test_mask.cpu().numpy()

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

With the dataset above, train three or four simple classifiers (e.g., SVM, LR, RF, KNN...) and report the accuracy score over the test set for them.

*Note: use **sklearn** for simplicity.*

## **B.2: Exploiting the Graph Structure**

Now, we are going to create a GNN with graph layers and a final linear layer. Refer to the Lab Statement for more details of how to implement the architecture. You should report your designed architecutre(s) and the accuracy results.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
# Remember to import the GNN modules you want to use


class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels): #You can change it at will (e.g., two different hidden channel sizes)
        super().__init__()

        # Your code here

    def forward(self, x, edge_index, return_embeddings=False):
        # Your code here

        return x


In [ ]:
# You are provided with a simple training and testing loop. Feel free to create your own improved versions.

gnn_model = GNN(dataset.num_node_features, 256, dataset.num_classes).to(device) #Remember to change it if you change the input parameters in the GNN class
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.01, weight_decay=5e-4)


def train_gnn(model):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test_gnn(model):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    correct = pred[data.test_mask] == data.y[data.test_mask]
    acc = int(correct.sum()) / int(data.test_mask.sum())
    return acc

for epoch in range(500):
    loss = train_gnn(gnn_model)

gnn_acc = test_gnn(gnn_model)
print("Test Accuracy:", gnn_acc)

## **B.4: The More the Merrier?**

Create new GNN models with 2, 4, 8 and 16 GNN layers and report the accuracy results. What do you observe? Why do you think it is happening?

## **B.5: What if we do  not have Initial Information?**

Train your previosuly best performing model with randomly initialized features and report the results:

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = Planetoid(root='data/PubMed', name='PubMed')
data = dataset[0]
data.x = #Replace the features for a 256 dimension random vector
data = data.to(device)

Now, we will see how to initialize the GNN with the embeddings resulting from a Knowledge Graph Embedding Model.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = Planetoid(root='data/PubMed', name='PubMed')
data = dataset[0]
data.x = #your_kge_embeddings (ensure they follow the same order!)
data = data.to(device)

Now, train again a GNN with the TransE embeddings as initialization:

In [ ]:
# Train

## **B.7: Where are the Embeddings?**

In this task we will obtaint the embeddings generated by our GNN architecture. First, update and retrain you best performing GNN model so that it returns the embeddings (see the lab statement for details).

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
# Remember to import the GNN/DL modules you want to use


class GNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels): #You can change it at will (e.g., two different hidden channel sizes)
        super().__init__()

        # Your code here

    def forward(self, x, edge_index, return_embeddings=False):
        # Your code here

        return x

gnn_model = GNN(dataset.num_node_features, 256, dataset.num_classes).to(device) #Remember to change it if you change the input parameters in the GNN class
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.01, weight_decay=5e-4)


def train_gnn(model):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(500):
    loss = train_gnn(gnn_model)

Obtain the embeddings:

In [ ]:
gnn_model.eval()
embs = gnn_model(data.x, data.edge_index, return_embs=True)
embs = embs.detach().cpu().numpy()

Once you have obtained the embeddings, apply dimensionality reduction (e.g., PCA) and visualize the result by coloring each embedding with its corresponding class.

## **B.8 Embedding spaces**